# 📘 Notebook 09: Supervised Learning I: Regression

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module C: Machine Learning · Notebook 2 of 4 in this module · (9 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Build and interpret a **linear regression** model
- Understand the **cost function** (Mean Squared Error) that defines a ‘good’ fit
- Implement **gradient descent** by hand to see how a model actually learns
- Train the same model the easy way with **scikit-learn**
- Extend to **polynomial regression** and connect it back to overfitting

> **Prerequisites:** Module C Notebook 08, plus NumPy and Matplotlib.
>
> 🔭 **Why gradient descent deserves your full attention:** it is the single optimisation algorithm that trains almost every model in this course, *including every neural network*. We implement it from scratch here, on the simplest possible model, so that when it reappears inside deep learning it is an old friend rather than a black box.

> ℹ️ **Setup note.** If needed: `import piplite; await piplite.install(['scikit-learn','numpy','matplotlib'])`

## 1. The problem: fitting a line

### Intuition first
Regression predicts a **continuous number**. The simplest case: we believe the target $y$ depends roughly linearly on a feature $x$, and we want the best straight line through the data. ‘Best’ will be made precise in a moment.

Let us create data with a known underlying relationship, $y = 3x + 2$, plus noise, so we can check whether our model recovers it:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.random.rand(100, 1) * 10          # 100 values in [0, 10]
y = 3 * X + 2 + np.random.randn(100, 1)  # true line y = 3x + 2, plus noise

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.7)
plt.title("Our regression data")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

## 2. The model and what makes a fit ‘good’

### The model
A line is described by two **parameters**: a slope $w$ (weight) and an intercept $b$ (bias):

$$\hat{y} = w x + b$$

$\hat{y}$ (‘y-hat’) is the model's **prediction**. Learning means finding the values of $w$ and $b$ that fit the data best.

### The cost function
To find the ‘best’ line we need to score how wrong a line is. We use the **Mean Squared Error (MSE)**, the average of the squared gaps between predictions and true values:

$$J(w, b) = \frac{1}{n} \sum_{i=1}^{n} \left( \hat{y}_i - y_i \right)^2 = \frac{1}{n} \sum_{i=1}^{n} \left( w x_i + b - y_i \right)^2$$

We **square** the errors so that positive and negative gaps do not cancel, and so that large errors are penalised heavily. Training the model means finding $w, b$ that make $J$ as small as possible.

In [ ]:
def predict(X, w, b):
    return w * X + b

def mse_cost(X, y, w, b):
    predictions = predict(X, w, b)
    return np.mean((predictions - y) ** 2)

# A deliberately bad guess vs a good guess
print("Cost with w=0, b=0:", mse_cost(X, y, 0, 0))
print("Cost with w=3, b=2:", mse_cost(X, y, 3, 2))

As expected, the line close to the true relationship (`w=3, b=2`) has a far lower cost. Now: how do we *find* those good values automatically?

## 3. Gradient descent: how the model learns

### Intuition first
Imagine standing on a foggy hillside, wanting to reach the valley floor (minimum cost). You cannot see far, but you can feel which direction slopes **downhill** and take a small step that way. Repeat, and you descend to the bottom. **Gradient descent** is exactly this: repeatedly step the parameters in the direction that reduces the cost.

### The mechanics
The **gradient** is the slope of the cost with respect to each parameter, it points uphill, so we move in the *opposite* direction. For MSE, calculus gives these gradients:

$$\frac{\partial J}{\partial w} = \frac{2}{n} \sum (\hat{y}_i - y_i)\, x_i \qquad \frac{\partial J}{\partial b} = \frac{2}{n} \sum (\hat{y}_i - y_i)$$

We then update each parameter by a small fraction of its gradient. That fraction is the **learning rate** $\alpha$:

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w} \qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

Let us implement this loop and watch it learn:

In [ ]:
def gradient_descent(X, y, w, b, learning_rate, epochs):
    n = len(X)
    history = []   # record the cost at each step
    for epoch in range(epochs):
        y_hat = predict(X, w, b)
        error = y_hat - y
        # gradients
        dw = (2/n) * np.sum(error * X)
        db = (2/n) * np.sum(error)
        # update step (move downhill)
        w = w - learning_rate * dw
        b = b - learning_rate * db
        history.append(mse_cost(X, y, w, b))
    return w, b, history

# Start from a bad guess and learn
w_final, b_final, history = gradient_descent(X, y, w=0.0, b=0.0,
                                             learning_rate=0.01, epochs=1000)

print(f"Learned w = {w_final:.3f} (true value 3)")
print(f"Learned b = {b_final:.3f} (true value 2)")

The algorithm recovered the underlying relationship without ever being told it. Let us visualise the learning process, the cost should fall and flatten:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: cost decreasing over epochs
axes[0].plot(history)
axes[0].set_title("Cost decreasing during training")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE cost")

# Right: the learned line over the data
axes[1].scatter(X, y, alpha=0.6, label="data")
x_line = np.linspace(0, 10, 100).reshape(-1, 1)
axes[1].plot(x_line, predict(x_line, w_final, b_final), color="red", label="learned line")
axes[1].set_title("The fitted model")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].legend()
plt.tight_layout()
plt.show()

> ⚠️ **Common pitfalls**
>
> - **Learning rate too high** → the steps overshoot the valley and the cost explodes (diverges). Try `learning_rate=0.1` above to see it.
> - **Learning rate too low** → learning is correct but painfully slow; it may not converge within your epoch budget.
> - Choosing the learning rate is the most common practical tuning task. There is no universal value, it depends on the problem.

## 4. The easy way: scikit-learn

Now that you understand what happens *inside*, you can appreciate how much a library hides. scikit-learn fits the same model in three lines, using the consistent `fit` / `predict` interface you will use for every model in this course:

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)            # training happens here

print(f"Slope (w):     {model.coef_[0][0]:.3f}")
print(f"Intercept (b): {model.intercept_[0]:.3f}")

# The same fit/predict pattern works for EVERY sklearn model
y_pred = model.predict(X)

> 🧠 **The `fit` / `predict` API is universal in scikit-learn.** Linear regression, decision trees, k-NN, all of them. Learn the pattern once and you can swap algorithms by changing a single line.

## 5. Evaluating the regression
Using the metrics from Notebook 08:

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)
print(f"MSE: {mse:.3f}")
print(f"R^2: {r2:.3f}   (closer to 1.0 is better)")

## 6. Polynomial regression: and overfitting revisited

Not all relationships are straight lines. **Polynomial regression** fits curves by adding powers of the feature ($x^2, x^3, \dots$) as extra inputs. But more flexibility brings the overfitting risk from Notebook 08 back into play, a high-degree polynomial can contort itself to pass through noise.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(1)
Xc = np.sort(np.random.rand(30, 1) * 6 - 3, axis=0)
yc = 0.5 * Xc**3 - Xc**2 + 2 + np.random.randn(30, 1) * 3

plt.figure(figsize=(7, 4))
plt.scatter(Xc, yc, color="black", s=20, label="data")
xs = np.linspace(-3, 3, 200).reshape(-1, 1)
for degree, color in zip([1, 3, 12], ["blue", "green", "red"]):
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    poly_model.fit(Xc, yc)
    plt.plot(xs, poly_model.predict(xs), color=color, label=f"degree {degree}")
plt.ylim(-30, 30)
plt.title("Polynomial regression at different degrees")
plt.xlabel("x"); plt.ylabel("y")
plt.legend()
plt.show()

The degree-1 line underfits the cubic shape; degree-3 captures it well (it matches the true form of the data); degree-12 overfits, wiggling to chase individual points. This is the bias-variance trade-off in action, choosing model complexity is a real decision.

---
## ✏️ Exercises

### Exercise 1
Re-run the hand-coded `gradient_descent` with `learning_rate=0.1`. What happens to the learned parameters and why? Then try `learning_rate=0.001` with the same 1000 epochs, does it converge as well?

In [ ]:
# Your experiment here:


<details><summary>💡 Show solution</summary>

```python
w1, b1, h1 = gradient_descent(X, y, 0.0, 0.0, learning_rate=0.1, epochs=1000)
print('lr=0.1  ->', w1, b1)   # likely diverges (nan / huge numbers)
w2, b2, h2 = gradient_descent(X, y, 0.0, 0.0, learning_rate=0.001, epochs=1000)
print('lr=0.001 ->', w2, b2)  # converges, but slower / not all the way
```

*lr=0.1 overshoots and diverges; lr=0.001 is stable but too slow to reach the optimum in 1000 epochs. This is the learning-rate trade-off first-hand.*
</details>

### Exercise 2
Use scikit-learn's `LinearRegression` to fit the data, then predict the value of `y` when `x = 7`. Compare it to what the true relationship $y = 3x + 2$ would give (i.e. 23).

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
model = LinearRegression().fit(X, y)
pred = model.predict(np.array([[7]]))
print('Predicted:', pred[0][0])
print('True line gives:', 3*7 + 2)
```
</details>

## 🔑 Key takeaways

- Linear regression fits $\hat{y} = wx + b$; the **MSE cost** $J(w,b)$ measures how wrong a fit is.
- **Gradient descent** minimises the cost by repeatedly stepping parameters downhill, scaled by the **learning rate**.
- This same optimisation loop trains essentially every model in the course, including neural networks.
- scikit-learn's universal **`fit` / `predict`** API hides this machinery behind a few lines.
- **Polynomial regression** fits curves but reintroduces the overfitting risk, complexity must be chosen carefully.

---
**Next:** *Notebook 10: Supervised Learning II: Classification*, where we predict categories and learn to evaluate with confusion matrices and ROC curves.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*